 Phase 6-7 — Candidate Triplet Search
**Kaggle | T4 ×2 | Python 3.11**  
Generates high-quality cross-dataset candidate correspondences and identifies promising triplets for circuit growth.


In [10]:
!pip install -q pandas==3.0.3

In [ ]:
import gc, time, pickle, warnings, itertools
import numpy as np
import pandas as pd
import networkx as nx
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path
from sklearn.preprocessing import RobustScaler
from sklearn.neighbors import NearestNeighbors
from scipy.spatial.distance import cosine
warnings.filterwarnings("ignore")

 ── Kaggle paths ──────────────────────────────────────────────────────────────
INPUT_DIR     = Path("/kaggle/input/datasets/jit007/flywire-phase5")
if not INPUT_DIR.exists():
    INPUT_DIR = Path("/kaggle/input/flywire")
if not INPUT_DIR.exists():
    INPUT_DIR = Path("/kaggle/input/flywire")
WORK_DIR      = Path("/kaggle/working")
EXPORT_DIR    = WORK_DIR / "results" / "exports"
CHECKPOINT_DIR= WORK_DIR / "results" / "checkpoints"
FIGURE_DIR    = WORK_DIR / "results" / "figures"
for d in [EXPORT_DIR, CHECKPOINT_DIR, FIGURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DATASET_FILES = {
    "BANC": "banc_626_edge_list.csv",
    "FAFB": "fafb_783_edge_list.csv",
    "MANC": "manc_1.2.1_edge_list.csv",
    "MAOL": "maol_1.1_edge_list.csv",
    "MCNS": "mcns_0.9_edge_list.csv",
}
PROCESS_ORDER = ["MANC", "MAOL", "BANC", "FAFB", "MCNS"]
print("Dirs ready. Input:", INPUT_DIR)


In [ ]:
print("=" * 60)
print("FILE DISCOVERY")
print("=" * 60)
missing = []
for name, fname in DATASET_FILES.items():
    p = INPUT_DIR / fname
    status = f"OK  {p.stat().st_size/1e6:.1f} MB" if p.exists() else "MISSING"
    print(f"  {name:<6} {fname:<35} {status}")
    if not p.exists(): missing.append(fname)

ckpt_path = INPUT_DIR / "phase5_checkpoint_kaggle_v2.pkl"
ck_status = f"OK  {ckpt_path.stat().st_size/1e6:.1f} MB" if ckpt_path.exists() else "MISSING"
print(f"  CKPT   phase5_checkpoint.pkl                {ck_status}")
if not ckpt_path.exists(): missing.append("phase5_checkpoint_kaggle_v2.pkl")

if missing:
    raise FileNotFoundError(f"Missing files: {missing}")
print("All input files found.")


In [ ]:
print("=" * 60)
print("LOADING PHASE 5 CHECKPOINT")
print("=" * 60)
with open(ckpt_path, "rb") as f:
    ckpt5 = pickle.load(f)

required = ["hub_nodes", "hub_thresholds", "fingerprint_dfs", "wl_label_dfs", "phase5_summary"]
for k in required:
    if k not in ckpt5:
        raise KeyError(f"Phase 5 checkpoint missing key: {k}")

hub_nodes       = ckpt5["hub_nodes"]
hub_thresholds  = ckpt5["hub_thresholds"]
fingerprint_dfs = ckpt5["fingerprint_dfs"]
wl_label_dfs    = ckpt5["wl_label_dfs"]
phase5_summary  = ckpt5["phase5_summary"]

print(f"\nDatasets loaded: {list(fingerprint_dfs.keys())}")
for ds, fp in fingerprint_dfs.items():
    print(f"  {ds:<6}  nodes={fp.shape[0]:>8,}  WL metrics available")
print("\nPhase 5 checkpoint validated")


In [ ]:
print("=" * 60)
print("RELOADING NETWORKX GRAPHS")
print("=" * 60)
graphs = {}
for name in PROCESS_ORDER:
    t0 = time.time()
    fpath = INPUT_DIR / DATASET_FILES[name]
    df_e  = pd.read_csv(fpath, usecols=[0, 1], header=0,
                        dtype={"pre_root_id": np.int64, "post_root_id": np.int64})
    df_e.columns = ["src", "dst"]
    G = nx.from_pandas_edgelist(df_e, "src", "dst", create_using=nx.DiGraph())
    graphs[name] = G
    n, e = G.number_of_nodes(), G.number_of_edges()
    print(f"  {name:<6}  nodes={n:>8,}  edges={e:>10,}  [{time.time()-t0:.1f}s]")
    del df_e; gc.collect()
print("\nAll graphs loaded")


In [ ]:
print("=" * 60)
print("STEP 1 -- DATASET FEATURE DISTRIBUTION ANALYSIS")
print("=" * 60)

 CHANGE 12 (OPTION B): Compare raw distributions before normalization.
 The previous mean-cosine similarity was uninformative because z-scoring centers
 every dataset at zero. We now report median, variance, and IQR per feature instead.

base_features = [
    "total_degree",
    "reciprocal_ratio",
    "hub_neighbor_count",
    "two_hop_size",
    "pagerank",
]

dist_records = []
for ds in PROCESS_ORDER:
    fp = fingerprint_dfs[ds].copy().fillna(0)
    for feat in base_features:
        if feat not in fp.columns:
            continue
        vals = fp[feat].values
        dist_records.append({
            "dataset":  ds,
            "feature":  feat,
            "median":   float(np.median(vals)),
            "variance": float(np.var(vals)),
            "p25":      float(np.percentile(vals, 25)),
            "p75":      float(np.percentile(vals, 75)),
            "iqr":      float(np.percentile(vals, 75) - np.percentile(vals, 25)),
        })

dist_df = pd.DataFrame(dist_records)
dist_df.to_csv(EXPORT_DIR / "dataset_feature_distributions.csv", index=False)

print("\n-- Feature Distribution Summary (raw values, before normalization) --")
for feat in base_features:
    sub = dist_df[dist_df["feature"] == feat].set_index("dataset")
    print(f"\n  {feat}:")
    print(f"  {'dataset':<8}  {'median':>12}  {'variance':>14}  {'IQR':>12}")
    for ds in PROCESS_ORDER:
        if ds in sub.index:
            r = sub.loc[ds]
            print(f"  {ds:<8}  {r['median']:>12.4f}  {r['variance']:>14.4f}  {r['iqr']:>12.4f}")

print(f"\nSaved: {EXPORT_DIR / 'dataset_feature_distributions.csv'}")
print("Distribution analysis complete")


In [ ]:
print("=" * 60)
print("STEP 2 -- FEATURE ENGINEERING + NORMALIZATION")
print("=" * 60)

 ── CHANGE 1: WL rarity features ─────────────────────────────────────────────
 Within each dataset: rarity_wl_k = 1 / freq_wl_k.
 NEVER store raw WL label integers as matching features.
 NEVER compare WL IDs across datasets.

WL_COLUMNS = ["wl_0", "wl_1", "wl_2", "wl_3"]    actual column names in wl_label_dfs
WL_RARITY_COLS = ["rarity_wl_0", "rarity_wl_1", "rarity_wl_2", "rarity_wl_3"]

wl_rarity_dfs = {}    {ds: DataFrame indexed by node_id, cols rarity_wl_0..3}

print("\n-- CHANGE 1: WL Rarity Statistics (within-dataset) --")
for ds in PROCESS_ORDER:
    wl_df = wl_label_dfs[ds].copy()
    node_col = wl_df.columns[0]    first column is the node identifier

    rarity_data = {}
    for wl_col, rar_col in zip(WL_COLUMNS, WL_RARITY_COLS):
        if wl_col in wl_df.columns:
            freq_map = wl_df[wl_col].map(wl_df[wl_col].value_counts())
            freq_vals = freq_map.values.astype(float)
             rarity = -log(freq).  Interpretation:
               freq=1   (singleton) -> rarity = -log(1) = 0.0   <- RAREST
               freq=1000 (common)  -> rarity = -log(1000) ≈ -6.9 <- MOST COMMON
             Higher (less negative) rarity = rarer label.
            rarity_data[rar_col] = -np.log(np.maximum(freq_vals, 1.0))
        else:
            rarity_data[rar_col] = np.zeros(len(wl_df))

    rar_df = pd.DataFrame(rarity_data, index=wl_df[node_col].values)
    rar_df.index.name = "node_id"
    wl_rarity_dfs[ds] = rar_df

    r3 = rar_df["rarity_wl_3"]   -log(freq): 0.0 = singleton (rarest), negative = common
     FIX 2: singletons have freq=1, so rarity = -log(1) = 0.0 exactly
    singleton_pct = np.isclose(r3, 0.0).mean() * 100
    print(
        f"  {ds:<6}  rarity_wl_3 (-log scale): "
        f"min={r3.min():.4f}  median={r3.median():.4f}  "
        f"max={r3.max():.4f}  singletons={singleton_pct:.1f}%"
    )

 ── CHANGE 2: Local topology features ────────────────────────────────────────
 Compute in_degree, out_degree, successor/predecessor counts, ratios
 directly from the already-loaded NetworkX directed graphs.

 topology_cols contains ONLY features not already in fingerprint_dfs.
 in_degree, out_degree, reciprocal_edge_count already exist in fingerprint_dfs
 and must NOT be re-added here to avoid duplicate-column join errors.
topology_cols = [
    "successor_ratio",
    "predecessor_ratio",
]

topology_dfs = {}    {ds: DataFrame indexed by node_id}

print("\n-- CHANGE 2: Local Topology Features --")
for ds in PROCESS_ORDER:
    G = graphs[ds]
    node_ids = list(G.nodes())

    in_deg  = np.array([G.in_degree(n)  for n in node_ids], dtype=float)
    out_deg = np.array([G.out_degree(n) for n in node_ids], dtype=float)
    total   = in_deg + out_deg

     FIX 3: successor_count == out_degree and predecessor_count == in_degree in a DiGraph.
     Removed both to eliminate duplicate features.

     reciprocal_edge_count: edges where both (u,v) and (v,u) exist
    rec_count = np.array(
        [sum(1 for nb in G.successors(n) if G.has_edge(nb, n)) for n in node_ids],
        dtype=float,
    )

    safe_total = np.maximum(total, 1.0)
    succ_ratio = out_deg / safe_total
    pred_ratio = in_deg  / safe_total

     Only store genuinely new features here — in_degree, out_degree, and
     reciprocal_edge_count already live in fingerprint_dfs and must not be
     re-added to avoid a duplicate-column ValueError on the join below.
    topology_dfs[ds] = pd.DataFrame(
        {
            "successor_ratio":   succ_ratio,
            "predecessor_ratio": pred_ratio,
        },
        index=pd.Index(node_ids, name="node_id"),
    )
    print(
        f"  {ds:<6}  nodes={len(node_ids):>8,}  "
        f"avg_in={in_deg.mean():.1f}  avg_out={out_deg.mean():.1f}  "
        f"avg_rec={rec_count.mean():.1f}"
    )

 ── CHANGE 3 & 4: Expanded feature set + RobustScaler normalization ─────────

features_to_use = [
     original structural fingerprint features
    "total_degree",
    "reciprocal_ratio",
    "hub_neighbor_count",
    "two_hop_size",
    "pagerank",
     local topology (FIX 3: successor_count/predecessor_count removed — identical to out/in_degree)
    "in_degree",
    "out_degree",
    "reciprocal_edge_count",
    "successor_ratio",
    "predecessor_ratio",
     WL rarity as -log(freq) — FIX 5: log scale for stability (no raw WL IDs)
    "rarity_wl_0",
    "rarity_wl_1",
    "rarity_wl_2",
    "rarity_wl_3",
]

normalized_fingerprint_dfs = {}

print("\n-- CHANGE 4: Normalizing expanded feature set --")
for ds in PROCESS_ORDER:
    fp  = fingerprint_dfs[ds].copy().fillna(0).set_index("node_id")
    top = topology_dfs[ds]
    rar = wl_rarity_dfs[ds]

     join topology and rarity columns onto fp (align on node_id index)
    fp = fp.join(top, how="left").fillna(0)
    fp = fp.join(rar, how="left").fillna(0)

     Downweight WL rarity features to reduce over-dependence on WL labels
    for c in [
        "rarity_wl_0",
        "rarity_wl_1",
        "rarity_wl_2",
        "rarity_wl_3",
    ]:
        if c in fp.columns:
            fp[c] *= 0.5

     select only the 16 features (missing columns -> 0)
    for col in features_to_use:
        if col not in fp.columns:
            fp[col] = 0.0

    scaler = RobustScaler()
    scaled = scaler.fit_transform(fp[features_to_use])

    normalized_fingerprint_dfs[ds] = pd.DataFrame(
        scaled, index=fp.index, columns=features_to_use
    )
    print(
        f"  {ds:<6}  normalized {fp.shape[0]:>8,} nodes  "
        f"x {len(features_to_use)} features"
    )

print("\nFeature normalization complete")
print(f"Feature set ({len(features_to_use)} features): {features_to_use}")


In [ ]:
print("=" * 60)
print("STEP 3 & 4 -- MUTUAL KNN CANDIDATE GENERATION")
print("=" * 60)

 CHANGE 7: configuration parameters
K_NEIGHBORS = 50
TOP_CANDIDATES_PER_NODE = 10   Reduced from 50: limits A-B x B-C cross-product to 100 combos/node (was 2500)
 Experimental values:
 50  = baseline
 100 = higher recall

candidate_matches = {}
pairs = list(itertools.combinations(PROCESS_ORDER, 2))

for d1, d2 in pairs:
    print(f"\nMatching {d1} <-> {d2} ...")
    t0 = time.time()

    df1 = normalized_fingerprint_dfs[d1]
    df2 = normalized_fingerprint_dfs[d2]

    nodes1 = df1.index.values
    nodes2 = df2.index.values
    arr1   = df1.values
    arr2   = df2.values

     CHANGE 5: Forward pass d1 -> d2
    nn_fwd = NearestNeighbors(n_neighbors=K_NEIGHBORS, metric="cosine", n_jobs=-1)
    nn_fwd.fit(arr2)
    dist_fwd, idx_fwd = nn_fwd.kneighbors(arr1)    shape (n1, K)

     CHANGE 5: Reverse pass d2 -> d1
    nn_rev = NearestNeighbors(n_neighbors=K_NEIGHBORS, metric="cosine", n_jobs=-1)
    nn_rev.fit(arr1)
    dist_rev, idx_rev = nn_rev.kneighbors(arr2)    shape (n2, K)

     Build reverse neighbor sets: for each d2 node j, which d1 nodes appear?
     idx_rev[j] gives the K nearest d1 indices for d2 node j
     So rev_sets[j] = set of d1-indices that consider j a mutual neighbor
    rev_sets = [set(idx_rev[j]) for j in range(len(nodes2))]

     Expand forward matches into long format
    n1 = len(nodes1)
    src_idx = np.repeat(np.arange(n1), K_NEIGHBORS)
    dst_idx = idx_fwd.flatten()
    fwd_dist = dist_fwd.flatten()

     Forward rank (1-based, per source node)
    fwd_rank = np.tile(np.arange(1, K_NEIGHBORS + 1), n1)

     Reverse rank: rank of src_idx in dst_idx's reverse neighbor list
    rev_rank_vals = np.full(len(src_idx), K_NEIGHBORS + 1, dtype=int)
    for k, (si, di) in enumerate(zip(src_idx, dst_idx)):
        rev_nbrs = list(idx_rev[di])
        if si in rev_nbrs:
            rev_rank_vals[k] = rev_nbrs.index(si) + 1    1-based

    rev_dist_vals = np.array([
        dist_rev[di, rev_nbrs.index(si)] if si in (rev_nbrs := list(idx_rev[di])) else np.nan
        for si, di in zip(src_idx, dst_idx)
    ])

     CHANGE 5: mutual_match flag
    mutual = np.array([
        si in rev_sets[di]
        for si, di in zip(src_idx, dst_idx)
    ])

     FIX 1: candidate_score uses fwd+rev distance sum (symmetric)
     For mutual matches: fwd_dist + rev_dist  (both distances are known)
     For non-mutual:     fwd_dist + 1.0       (1.0 as penalty for absent reverse)
     np.nan_to_num handles cases where reverse_distance is missing (nan)
    rev_dist_filled = np.nan_to_num(rev_dist_vals, nan=1.0)
    cand_score = np.where(
        mutual,
        fwd_dist + rev_dist_filled,    mutual: true bidirectional distance sum
        fwd_dist + 1.0,                non-mutual: penalise absent reverse confirmation
    )

     FIX 3: rarity-difference penalty removed.
     WL rarity already participates in the KNN cosine distance via the normalized
     feature vectors.  Adding it again here would double-count its influence.
     candidate_score is purely: fwd+rev (mutual) or fwd+1.0 (non-mutual).

    matches_df = pd.DataFrame({
        "node_1":          nodes1[src_idx],
        "node_2":          nodes2[dst_idx],
        "distance":        fwd_dist,                original forward cosine distance
        "forward_distance": fwd_dist,
        "reverse_distance": rev_dist_vals,
        "forward_rank":    fwd_rank,
        "reverse_rank":    rev_rank_vals,
        "mutual_match":    mutual,
        "candidate_score": cand_score,              CHANGE 6: sort key
    })

     CHANGE 6: sort by candidate_score (lower is better)
    matches_df = matches_df.sort_values("candidate_score").reset_index(drop=True)
    matches_df.to_csv(EXPORT_DIR / f"candidates_{d1}_{d2}.csv", index=False)
    candidate_matches[f"{d1}_{d2}"] = matches_df

    elapsed = time.time() - t0
    n_mutual = mutual.sum()

    avg_candidates = len(matches_df) / max(len(nodes1), 1)

    print(
        f"  Average candidates per node: "
        f"{avg_candidates:.1f}"
    )

    print(
        f"  Mutual candidates: "
        f"{n_mutual:,}"
    )
    print(
        f"  Found {len(matches_df):,} candidates in {elapsed:.1f}s  "
        f"mutual={n_mutual:,} ({100*n_mutual/len(matches_df):.1f}%)"
    )
    gc.collect()

print("\nCandidate generation complete")
print(f"\nK_NEIGHBORS={K_NEIGHBORS}  TOP_CANDIDATES_PER_NODE={TOP_CANDIDATES_PER_NODE}")
print("\n-- Candidate Counts --")
for key, df in candidate_matches.items():
    mut_pct = 100 * df["mutual_match"].mean()
    print(f"  {key:<20}  {len(df):>10,} pairs  mutual={mut_pct:.1f}%")


In [ ]:
print("=" * 60)
print("STEP 5 -- TRIPLET GENERATION (A-B-C + A-C VALIDATION)")
print("=" * 60)

triplets = list(itertools.combinations(PROCESS_ORDER, 3))
triplet_candidates = {}


def _get_candidates(d_src, d_dst):
    """Return candidate DataFrame for a dataset pair in canonical (src->dst) orientation."""
    key_fwd = f"{d_src}_{d_dst}"
    key_rev = f"{d_dst}_{d_src}"
    if key_fwd in candidate_matches:
        return candidate_matches[key_fwd].copy()
    if key_rev in candidate_matches:
        df = candidate_matches[key_rev].copy()
        return df.rename(columns={"node_1": "node_2", "node_2": "node_1"})
    return pd.DataFrame()


for trip in triplets:
    d1, d2, d3 = trip
    print(f"\nGenerating triplet: {d1}-{d2}-{d3}")

    m12 = _get_candidates(d1, d2)
    m23 = _get_candidates(d2, d3)
    m13 = _get_candidates(d1, d3)    CHANGE 8: A-C relationship

    if m12.empty or m23.empty:
        print("  Missing pairwise matches (12 or 23), skipping")
        triplet_candidates[trip] = pd.DataFrame()
        continue

    if m13.empty:
        print("  Warning: no A-C candidates found — applying soft penalty")

     CHANGE 7: use TOP_CANDIDATES_PER_NODE
    m12_top = (
        m12.sort_values("candidate_score")
           .groupby("node_1").head(TOP_CANDIDATES_PER_NODE)
           .reset_index(drop=True)
    )
    m23_top = (
        m23.sort_values("candidate_score")
           .groupby("node_1").head(TOP_CANDIDATES_PER_NODE)
           .reset_index(drop=True)
    )

     Merge A-B with B-C on the middle node
    merged = pd.merge(
        m12_top[["node_1", "node_2", "candidate_score"]].rename(
            columns={"candidate_score": "distance_12"}
        ),
        m23_top[["node_1", "node_2", "candidate_score"]].rename(
            columns={"node_1": "node_2_mid", "node_2": "node_3", "candidate_score": "distance_23"}
        ),
        left_on="node_2",
        right_on="node_2_mid",
    ).drop(columns=["node_2_mid"])

    if merged.empty:
        print("  No triplets formed after A-B-C merge")
        triplet_candidates[trip] = pd.DataFrame()
        continue

    print(f"  A-B-C raw triplets: {len(merged):,}")

     CHANGE 8: Soft A-C validation — preserve triplets and penalize missing A-C support
    if not m13.empty:
        m13_top = (
            m13.sort_values("candidate_score")
               .groupby("node_1").head(TOP_CANDIDATES_PER_NODE)
               .reset_index(drop=True)
        )[["node_1", "node_2", "candidate_score"]].rename(
            columns={"node_2": "node_3", "candidate_score": "distance_13"}
        )
         CHANGE 9: left join — soft A-C validation (preserve recall)
        merged = pd.merge(merged, m13_top, on=["node_1", "node_3"], how="left")
        merged["ac_supported"] = merged["distance_13"].notna()
        merged["distance_13"] = merged["distance_13"].fillna(1.0)
        print(f"  After A-C validation: {len(merged):,}")
    else:
        merged["ac_supported"] = False
        merged["distance_13"] = 1.0

    if merged.empty:
        triplet_candidates[trip] = pd.DataFrame()
        continue

     Conservative filtering: remove triplets with any duplicate node IDs
    dup_mask = (
        (merged["node_1"] == merged["node_2"]) |
        (merged["node_2"] == merged["node_3"]) |
        (merged["node_1"] == merged["node_3"])
    )
    merged = merged[~dup_mask & merged["node_1"].notna() &
                    merged["node_2"].notna() & merged["node_3"].notna()].reset_index(drop=True)

     CHANGE 3: graph-aware ranking — reward A-C supported triplets without hard filtering
    graph_bonus = merged["ac_supported"].astype(float)
    merged["triplet_score"] = (
        merged["distance_12"] + merged["distance_23"] + 0.5 * merged["distance_13"]
        - 0.25 * graph_bonus
    )

     CHANGE 5: A-C support rate diagnostic
    ac_rate = merged["ac_supported"].mean()
    print(f"  A-C support rate: {100 * ac_rate:.1f}%")

     Backward-compatible alias expected by Phase 8
    merged["similarity_score"] = merged["triplet_score"]

     CHANGE 11: deduplicate (node_1, node_2, node_3) — keep lowest triplet_score
    merged = (
        merged.sort_values("triplet_score")
              .drop_duplicates(subset=["node_1", "node_2", "node_3"], keep="first")
              .reset_index(drop=True)
    )

     CHANGE 6: per-node triplet pruning — retain best-scoring triplets per source node
     Dramatically reduces RAM / checkpoint size while preserving top candidates.
    MAX_TRIPLETS_PER_NODE = 10
    merged = (
        merged
        .sort_values("triplet_score")
        .groupby("node_1", group_keys=False)
        .head(MAX_TRIPLETS_PER_NODE)
        .reset_index(drop=True)
    )

    print(f"  Final viable seeds: {len(merged):,}")
    triplet_candidates[trip] = merged

 Combine all triplet seeds for export (CHANGE 13: preserve expected filenames/columns)
all_trip_seeds = []
for trip, df in triplet_candidates.items():
    if df.empty:
        continue
    df = df.copy()
    df["dataset1"] = trip[0]
    df["dataset2"] = trip[1]
    df["dataset3"] = trip[2]
    all_trip_seeds.append(df)

if all_trip_seeds:
    final_triplets_df = pd.concat(all_trip_seeds, ignore_index=True)
     Ensure similarity_score column present for Phase 8 compatibility
    if "similarity_score" not in final_triplets_df.columns:
        final_triplets_df["similarity_score"] = final_triplets_df["triplet_score"]
    final_triplets_df.to_csv(EXPORT_DIR / "triplet_seed_candidates.csv", index=False)
    print(f"\nExported all triplet seeds ({len(final_triplets_df):,})")
    print(f"Columns: {list(final_triplets_df.columns)}")
else:
    final_triplets_df = pd.DataFrame()
    print("\nNo triplet seeds generated.")


In [ ]:
print("=" * 60)
print("STEP 6 -- TRIPLET SCORING + DIAGNOSTICS")
print("=" * 60)

triplet_ranking = []
for trip, df in triplet_candidates.items():
    d1, d2, d3 = trip
    if len(df) == 0:
        continue
    triplet_ranking.append({
        "triplet":         f"{d1}-{d2}-{d3}",
        "viable_seeds":    len(df),
        "avg_triplet_score": df["triplet_score"].mean(),
        "top_triplet_score": df["triplet_score"].min(),
        "mutual_pair_pct": float("nan"),   informational placeholder
    })

rank_df = (
    pd.DataFrame(triplet_ranking)
      .sort_values("avg_triplet_score")
      .reset_index(drop=True)
)
rank_df.to_csv(EXPORT_DIR / "triplet_ranking.csv", index=False)

fig = px.bar(
    rank_df, x="triplet", y="viable_seeds",
    title="Viable Seeds per Triplet", template="plotly_dark"
)
fig.write_html(str(FIGURE_DIR / "triplet_ranking.html"))

print("\n-- Triplet Family Rankings (lower avg_triplet_score = better) --")
print(rank_df.to_string(index=False))

if not final_triplets_df.empty:
    print("\n-- Top 20 Triplets by triplet_score --")
    top20_cols = [c for c in [
        "node_1", "node_2", "node_3",
        "triplet_score", "distance_12", "distance_23", "distance_13",
        "similarity_score", "dataset1", "dataset2", "dataset3",
    ] if c in final_triplets_df.columns]
    top20 = final_triplets_df.sort_values("triplet_score").head(20)[top20_cols]
    print(top20.to_string(index=False))

    print("\n-- Top Candidate Families (by seed count) --")
    fam_counts = (
        final_triplets_df
        .groupby(["dataset1", "dataset2", "dataset3"])
        .size()
        .reset_index(name="n_seeds")
        .sort_values("n_seeds", ascending=False)
    )
    print(fam_counts.to_string(index=False))

     Mutual match statistics across all candidate pairs
    print("\n-- Mutual Match Statistics across candidate_matches --")
    for key, df in candidate_matches.items():
        if "mutual_match" in df.columns:
            pct = 100 * df["mutual_match"].mean()
            print(f"  {key:<22}  mutual={pct:.1f}%  total={len(df):,}")

print("\nTriplet scoring complete")


In [ ]:
print("=" * 60)
print("SAVING PHASE 6-7 CHECKPOINT")
print("=" * 60)

 CHANGE 13: all keys that Phase 8-10 depends on are preserved unchanged.
 New keys are strictly additive.
phase6_7_checkpoint = {
     ── backward-compatible keys (Phase 8-10 reads these) ─────────────────────
    "hub_nodes":                  hub_nodes,
    "hub_thresholds":             hub_thresholds,
    "fingerprint_dfs":            fingerprint_dfs,
    "wl_label_dfs":               wl_label_dfs,
    "normalized_fingerprint_dfs": normalized_fingerprint_dfs,
    "candidate_matches":          candidate_matches,
    "triplet_candidates":         triplet_candidates,
    "triplet_ranking":            rank_df,
     ── new extended keys (additive — Phase 8 can optionally use) ─────────────
    "wl_rarity_dfs":              wl_rarity_dfs,
    "topology_dfs":               topology_dfs,
    "phase6_7_config": {
        "K_NEIGHBORS":             K_NEIGHBORS,
        "TOP_CANDIDATES_PER_NODE": TOP_CANDIDATES_PER_NODE,
        "MAX_TRIPLETS_PER_NODE":   10,
        "features_to_use":         features_to_use,
        "mutual_knn":              True,
        "ac_validation":           True,
        "triplet_score":           "distance_12 + distance_23 + 0.5 * distance_13 - 0.25 * ac_supported",
        "soft_ac_validation":    True,
        "ac_weight":             0.5,
        "graph_bonus_weight":    0.25,
        "scaler":                "RobustScaler",
    },
}

ckpt_out = CHECKPOINT_DIR / "phase6_7_checkpoint.pkl"
with open(ckpt_out, "wb") as f:
    pickle.dump(phase6_7_checkpoint, f)
print(f"  Saved: {ckpt_out}  ({ckpt_out.stat().st_size / 1e6:.1f} MB)")
print("Phase 6-7 checkpoint saved")


In [ ]:
print("=" * 60)
print("VALIDATION REPORT")
print("=" * 60)
ok = True

 FIX 4: replaced dataset_similarity_matrix.csv with dataset_feature_distributions.csv
for fname in ["dataset_feature_distributions.csv", "triplet_ranking.csv", "triplet_seed_candidates.csv"]:
    p = EXPORT_DIR / fname
    if not p.exists():
        print(f"  MISSING: {fname}"); ok = False
    else:
        print(f"  OK: {fname}")

for p1, p2 in pairs:
    p = EXPORT_DIR / f"candidates_{p1}_{p2}.csv"
    if not p.exists():
        print(f"  MISSING: candidates_{p1}_{p2}.csv"); ok = False

if not (CHECKPOINT_DIR / "phase6_7_checkpoint.pkl").exists():
    print("  MISSING: phase6_7_checkpoint.pkl"); ok = False

print()
if ok:
    print("All validation checks passed")
else:
    print("Validation failures detected — see above")


In [ ]:
print("=" * 60)
print("PHASE 6–7 COMPLETE")
print("READY FOR PHASE 8 CIRCUIT GROWTH")
print("=" * 60)


In [ ]:
import shutil

shutil.make_archive(
    "/kaggle/working/flywire_results",
    "zip",
    "/kaggle/working/results"
)

print("ZIP created")